#### 철새 도래지 거리 및 도래지 내/인근
##### # 좌표계: 철새도래지 EPSG:5179(TM) → EPSG:4326(WGS84) 변환 후 haversine 거리 계산


In [ ]:
!pip install transformers
%pip install pyproj
%pip install pyproj geopandas
%pip install azure-storage-blob

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG,BASE_PATH,SECRET_SCOPE

import os
import zipfile, io
from azure.storage.blob import BlobServiceClient

import hashlib
import numpy as np
import geopandas as gpd
from pyproj import Transformer

In [ ]:
# ── 출력 테이블 ──── 
OUTPUT_TABLE = f"{CATALOG}.gold.farm_bird_habitat_daily"

In [ ]:
# ── 철새도래지 shapefile: Blob → Workspace 복사 후 로드 ───────

# Blob 으로 읽기
os.makedirs("/Workspace/방역로/tmp/bird_habitat", exist_ok=True)

for ext in ["shp", "dbf", "shx", "prj"]:
    dbutils.fs.cp(
        f"{BASE_PATH}/habitat/A2SM_MGRBIRDSHBTT.{ext}",
        f"file:///Workspace/방역로/tmp/bird_habitat/A2SM_MGRBIRDSHBTT.{ext}"
    )

bird = gpd.read_file("/Workspace/방역로/tmp/bird_habitat/A2SM_MGRBIRDSHBTT.shp").to_crs("EPSG:4326")
bird_coords = np.array([[geom.centroid.x, geom.centroid.y] for geom in bird.geometry])  # (N, 2) lon, lat


In [ ]:
# ── 데이터 로드 ──────────────────────────────────────────────
farm = spark.read.table(f"{CATALOG}.silver.farm_master").toPandas()

In [ ]:
# ── Haversine 거리 계산 (벡터화) ────────────────────────────
def haversine_min_km(farm_lat, farm_lon):
    """농장 1개 기준 모든 도래지까지 거리(km) 배열 반환"""
    R = 6371.0
    lat1, lon1 = np.radians(farm_lat), np.radians(farm_lon)
    lat2 = np.radians(bird_coords[:, 1])
    lon2 = np.radians(bird_coords[:, 0])
    a = np.sin((lat2 - lat1)/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1)/2)**2
    return R * 2 * np.arcsin(np.sqrt(a).clip(0, 1))


In [ ]:
farm["farm_id"] = farm.apply(
    lambda r: hashlib.md5(f"{r['farm_name']}|{r['farm_address']}".encode()).hexdigest(), axis=1
)

dists = farm.apply(lambda r: haversine_min_km(r["latitude"], r["longitude"]).min(), axis=1)

farm["farm_bird_nearest_dist_km"]       = dists.round(4)
farm["within_migratory_bird_site_10km"] = (dists <= 10.0).astype(int)  # ← 변경: bool → int(0/1)


In [ ]:
# ── 결과 저장 ────────────────────────────────────────────────
result = farm[["farm_id", "farm_name", "farm_address", "farm_bird_nearest_dist_km", "within_migratory_bird_site_10km"]].drop_duplicates(subset=["farm_id"])

(
    spark.createDataFrame(result)
    .writeTo(OUTPUT_TABLE)
    .using("delta")
    .createOrReplace()
)